# Feature Analysis — Fingerprint Spoofing Detection

This notebook is the first stage of a project on **fingerprint spoofing detection**: a
binary classification task where the goal is to distinguish genuine fingerprint images
(class `1`) from spoofed/fake ones (class `0`), based on a 6-dimensional feature vector
produced by an image feature extractor.

Here we perform exploratory data analysis on the training set: per-class, per-feature
histograms and pairwise scatter plots, used to characterize class overlap, modality, and
cluster structure before any model is trained. These observations directly motivate the
modeling choices made in the later stages of the project (dimensionality reduction,
Gaussian classifiers, logistic regression, SVMs, GMMs).

**Dataset.** `data/trainData.txt` is a CSV file with one sample per row: the first 6
columns are the features, and the last column is the class label (`0` = fake,
`1` = genuine).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

## 1. Load the data

In [ ]:
def load(file_path):
    """
    Load the project dataset from a CSV file.

    Each row is a sample: the first 6 comma-separated values are the
    features, and the last value is the integer class label (0 = fake,
    1 = genuine).

    Returns
    -------
    data : ndarray of shape (6, n_samples)
        Feature matrix, features as rows and samples as columns.
    labels : ndarray of shape (n_samples,)
        Integer class labels.
    """
    with open(file_path, "r") as f:
        lines = f.readlines()

    data, labels = [], []
    for line in lines:
        *features_str, label_str = line.strip().split(",")
        data.append([float(x) for x in features_str])
        labels.append(int(label_str))

    return np.array(data).T, np.array(labels)

In [ ]:
D, L = load("../Data/trainData.txt")
D.shape, L.shape

In [ ]:
# Split the data by class for per-class analysis
D0 = D[:, L == 0]  # fake
D1 = D[:, L == 1]  # genuine

D0.shape, D1.shape

## 2. Per-feature histograms

Normalized histograms of each feature, split by class, give a first look at class
overlap, relative means/variances, and the number of modes per feature.

In [ ]:
feature_labels = [
    "Feature 1",
    "Feature 2",
    "Feature 3",
    "Feature 4",
    "Feature 5",
    "Feature 6",
]

fig, axs = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Feature histograms by class", fontsize=18)
axs = axs.ravel()

for i in range(6):
    axs[i].hist(D0[i, :], label="Fake", density=True, alpha=0.5, bins=100)
    axs[i].hist(D1[i, :], label="Real", density=True, alpha=0.5, bins=100)
    axs[i].set_xlabel(feature_labels[i])
    axs[i].set_ylabel("Density")
    axs[i].legend()

plt.tight_layout()
plt.show()

## 3. Pairwise scatter plots

All $\binom{6}{2}=15$ pairwise scatter plots, to check for structure (clusters,
non-linear boundaries) that is not visible in the 1D histograms above.

In [ ]:
fig, axs = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("Pairwise scatter plots (all feature combinations)", fontsize=18)
axs = axs.flatten()

plot_idx = 0
for i in range(6):
    for j in range(i + 1, 6):
        ax = axs[plot_idx]
        ax.scatter(D0[i, :], D0[j, :], label="Fake", alpha=0.5)
        ax.scatter(D1[i, :], D1[j, :], label="Real", alpha=0.5)
        ax.set_xlabel(feature_labels[i])
        ax.set_ylabel(feature_labels[j])
        if plot_idx == 0:
            ax.legend()
        plot_idx += 1

plt.tight_layout()
plt.show()

**Why do the histograms show only 2–3 peaks for features 5 and 6, while the scatter
plot reveals 4 separable clusters per pair?**

A histogram is a projection of the data onto a single axis. Two clusters that happen to
share the same position along that axis collapse into a single histogram peak, even
though they are clearly separated in the full 2D (or higher-dimensional) view:

- For the fake class, 2 histogram peaks correspond to 2 *pairs* of clusters (4 in total)
  that overlap along that one axis.
- For the genuine class, 3 histogram peaks appear because the central peak collects two
  clusters that overlap along that axis.

This is a useful general caution: 1D histograms and 2D scatter plots only show what is
visible from a given "viewing angle" — they can under-report the true cluster structure
of a higher-dimensional dataset.

## 4. Per-class covariance and mean separation

To complement the visual analysis, we compute the per-class covariance matrices and the
absolute difference between class means for each feature.

In [ ]:
def covariance_matrix(D):
    """Compute the empirical covariance matrix of a (n_features, n_samples) data matrix."""
    mu = D.mean(axis=1, keepdims=True)
    D_centered = D - mu
    return (D_centered @ D_centered.T) / D.shape[1]


mean_D0 = D0.mean(axis=1, keepdims=True)
mean_D1 = D1.mean(axis=1, keepdims=True)

cov_D0 = covariance_matrix(D0)
cov_D1 = covariance_matrix(D1)

cov_D0

In [ ]:
cov_D1

In [ ]:
# Absolute difference between per-feature class means
abs(mean_D0 - mean_D1)

## 5. Feature-by-feature analysis

### 5.1 Features 1 and 2

- **Class overlap.** Strong overlap between the fake and genuine classes: both
  distributions are concentrated around the center, and in the scatter plot the two
  point clouds are highly mixed.
- **Means.** Close to identical for both features — no meaningful shift between classes.
- **Variances.** Not equal: for feature 1 the genuine class is more spread out (larger
  variance) than the fake class; for feature 2 the fake class is the more spread-out one.
  The two features effectively swap which class is "wide" and which is "narrow".
- **Modality.** Both features are unimodal (one dominant peak per class).

Because the class-conditional distributions have almost identical means but different
variances, a *linear* classifier is not expected to work well on these two features
alone — the discriminative signal is in the spread, not in the location.

### 5.2 Features 3 and 4

- **Class overlap.** Much better separated than features 1 and 2: the fake class is
  shifted toward negative values, the genuine class toward positive values. Some overlap
  remains in the tails, where the two distributions meet.
- **Means.** Clearly different between classes (this is the main source of separation).
- **Variances.** Similar between classes for both features (both around $0.55$).
- **Modality.** Both unimodal.

This is close to the textbook case for a linear classifier: similar within-class spread,
well-separated means. A linear decision boundary should be able to separate the two
classes reasonably well using these two features.

### 5.3 Features 5 and 6

- **Class overlap.** The most structured pair in the dataset. The scatter plot shows a
  grid-like / checkerboard pattern rather than a single compact cloud per class; the two
  classes still overlap, but in a structured way, concentrated in different local
  "islands" of the joint feature space.
- **Modality.** Both features are clearly multimodal (roughly 2–3 peaks per class in the
  1D histograms).
- **Clusters.** The scatter plot of feature 5 vs. feature 6 reveals at least 4 separable
  clusters (2 per class), interleaved with each other — substantially more structure than
  is visible from the 1D histograms alone (see the discussion in Section 3).

This multi-modal, multi-cluster structure means that a simple linear classifier, or a
single-Gaussian model, is expected to be a poor fit for these two features individually
— a theme this analysis returns to in later stages of the project (density fitting,
Gaussian classifiers, GMMs).